In [0]:
df = spark.read.parquet("/Volumes/workspace/default/citibike_data/citibike_weather_joined_202406.parquet")

In [0]:
df.printSchema()

root
 |-- ride_id: string (nullable = true)
 |-- rideable_type: string (nullable = true)
 |-- started_at: string (nullable = true)
 |-- ended_at: string (nullable = true)
 |-- start_station_name: string (nullable = true)
 |-- start_station_id: string (nullable = true)
 |-- end_station_name: string (nullable = true)
 |-- end_station_id: string (nullable = true)
 |-- start_lat: string (nullable = true)
 |-- start_lng: string (nullable = true)
 |-- end_lat: string (nullable = true)
 |-- end_lng: string (nullable = true)
 |-- member_casual: string (nullable = true)
 |-- trip_date: string (nullable = true)
 |-- STATION: string (nullable = true)
 |-- LATITUDE: string (nullable = true)
 |-- LONGITUDE: string (nullable = true)
 |-- ELEVATION: string (nullable = true)
 |-- NAME: string (nullable = true)
 |-- TEMP: string (nullable = true)
 |-- TEMP_ATTRIBUTES: string (nullable = true)
 |-- DEWP: string (nullable = true)
 |-- DEWP_ATTRIBUTES: string (nullable = true)
 |-- SLP: string (nullable =

# **Basic Sanity Checks**


**Number of Rows & Columns**


In [0]:
print("Total Rows:", df.count())
print("Total Columns:", len(df.columns))

Total Rows: 4783576
Total Columns: 41


**Column list**

In [0]:
df.columns

['ride_id',
 'rideable_type',
 'started_at',
 'ended_at',
 'start_station_name',
 'start_station_id',
 'end_station_name',
 'end_station_id',
 'start_lat',
 'start_lng',
 'end_lat',
 'end_lng',
 'member_casual',
 'trip_date',
 'STATION',
 'LATITUDE',
 'LONGITUDE',
 'ELEVATION',
 'NAME',
 'TEMP',
 'TEMP_ATTRIBUTES',
 'DEWP',
 'DEWP_ATTRIBUTES',
 'SLP',
 'SLP_ATTRIBUTES',
 'STP',
 'STP_ATTRIBUTES',
 'VISIB',
 'VISIB_ATTRIBUTES',
 'WDSP',
 'WDSP_ATTRIBUTES',
 'MXSPD',
 'GUST',
 'MAX',
 'MAX_ATTRIBUTES',
 'MIN',
 'MIN_ATTRIBUTES',
 'PRCP',
 'PRCP_ATTRIBUTES',
 'SNDP',
 'FRSHTT']

In [0]:
from pyspark.sql.functions import col, to_timestamp, to_date

df_clean = df \
.withColumn("started_at", to_timestamp("started_at")) \
.withColumn("ended_at", to_timestamp("ended_at")) \
.withColumn("trip_date", to_date("trip_date")) \
.withColumn("start_lat", col("start_lat").cast("double")) \
.withColumn("start_lng", col("start_lng").cast("double")) \
.withColumn("end_lat", col("end_lat").cast("double")) \
.withColumn("end_lng", col("end_lng").cast("double")) \
.withColumn("LATITUDE", col("LATITUDE").cast("double")) \
.withColumn("LONGITUDE", col("LONGITUDE").cast("double")) \
.withColumn("ELEVATION", col("ELEVATION").cast("double")) \
.withColumn("TEMP", col("TEMP").cast("double")) \
.withColumn("DEWP", col("DEWP").cast("double")) \
.withColumn("SLP", col("SLP").cast("double")) \
.withColumn("STP", col("STP").cast("double")) \
.withColumn("VISIB", col("VISIB").cast("double")) \
.withColumn("WDSP", col("WDSP").cast("double")) \
.withColumn("MXSPD", col("MXSPD").cast("double")) \
.withColumn("GUST", col("GUST").cast("double")) \
.withColumn("MAX", col("MAX").cast("double")) \
.withColumn("MIN", col("MIN").cast("double")) \
.withColumn("PRCP", col("PRCP").cast("double")) \
.withColumn("SNDP", col("SNDP").cast("double")) \
.withColumn("STATION", col("STATION").cast("string"))


In [0]:
df_clean.printSchema()


root
 |-- ride_id: string (nullable = true)
 |-- rideable_type: string (nullable = true)
 |-- started_at: timestamp (nullable = true)
 |-- ended_at: timestamp (nullable = true)
 |-- start_station_name: string (nullable = true)
 |-- start_station_id: string (nullable = true)
 |-- end_station_name: string (nullable = true)
 |-- end_station_id: string (nullable = true)
 |-- start_lat: double (nullable = true)
 |-- start_lng: double (nullable = true)
 |-- end_lat: double (nullable = true)
 |-- end_lng: double (nullable = true)
 |-- member_casual: string (nullable = true)
 |-- trip_date: date (nullable = true)
 |-- STATION: string (nullable = true)
 |-- LATITUDE: double (nullable = true)
 |-- LONGITUDE: double (nullable = true)
 |-- ELEVATION: double (nullable = true)
 |-- NAME: string (nullable = true)
 |-- TEMP: double (nullable = true)
 |-- TEMP_ATTRIBUTES: string (nullable = true)
 |-- DEWP: double (nullable = true)
 |-- DEWP_ATTRIBUTES: string (nullable = true)
 |-- SLP: double (nullab

In [0]:
print("Total Rows:", df_clean.count())
print("Total Columns:", len(df_clean.columns))


Total Rows: 4783576
Total Columns: 41


In [0]:
df_clean.show(5)

+----------------+-------------+--------------------+--------------------+--------------------+----------------+--------------------+--------------+-----------------+------------------+-----------------+------------------+-------------+----------+-----------+--------+---------+---------+--------------------+----+---------------+----+---------------+------+--------------+----+--------------+-----+----------------+----+---------------+-----+-----+----+--------------+----+--------------+----+---------------+-----+------+
|         ride_id|rideable_type|          started_at|            ended_at|  start_station_name|start_station_id|    end_station_name|end_station_id|        start_lat|         start_lng|          end_lat|           end_lng|member_casual| trip_date|    STATION|LATITUDE|LONGITUDE|ELEVATION|                NAME|TEMP|TEMP_ATTRIBUTES|DEWP|DEWP_ATTRIBUTES|   SLP|SLP_ATTRIBUTES| STP|STP_ATTRIBUTES|VISIB|VISIB_ATTRIBUTES|WDSP|WDSP_ATTRIBUTES|MXSPD| GUST| MAX|MAX_ATTRIBUTES| MIN|MI

# **Missing Value Analysis**

In [0]:
# Missing values per column
from pyspark.sql.functions import count, when
df_clean.select([count(when(col(c).isNull(),c)).alias(c)
                 for c in df_clean.columns]).show()


+-------+-------------+----------+--------+------------------+----------------+----------------+--------------+---------+---------+-------+-------+-------------+---------+-------+--------+---------+---------+----+----+---------------+----+---------------+---+--------------+---+--------------+-----+----------------+----+---------------+-----+----+---+--------------+---+--------------+----+---------------+----+------+
|ride_id|rideable_type|started_at|ended_at|start_station_name|start_station_id|end_station_name|end_station_id|start_lat|start_lng|end_lat|end_lng|member_casual|trip_date|STATION|LATITUDE|LONGITUDE|ELEVATION|NAME|TEMP|TEMP_ATTRIBUTES|DEWP|DEWP_ATTRIBUTES|SLP|SLP_ATTRIBUTES|STP|STP_ATTRIBUTES|VISIB|VISIB_ATTRIBUTES|WDSP|WDSP_ATTRIBUTES|MXSPD|GUST|MAX|MAX_ATTRIBUTES|MIN|MIN_ATTRIBUTES|PRCP|PRCP_ATTRIBUTES|SNDP|FRSHTT|
+-------+-------------+----------+--------+------------------+----------------+----------------+--------------+---------+---------+-------+-------+-------------

In [0]:
df_clean.columns

['ride_id',
 'rideable_type',
 'started_at',
 'ended_at',
 'start_station_name',
 'start_station_id',
 'end_station_name',
 'end_station_id',
 'start_lat',
 'start_lng',
 'end_lat',
 'end_lng',
 'member_casual',
 'trip_date',
 'STATION',
 'LATITUDE',
 'LONGITUDE',
 'ELEVATION',
 'NAME',
 'TEMP',
 'TEMP_ATTRIBUTES',
 'DEWP',
 'DEWP_ATTRIBUTES',
 'SLP',
 'SLP_ATTRIBUTES',
 'STP',
 'STP_ATTRIBUTES',
 'VISIB',
 'VISIB_ATTRIBUTES',
 'WDSP',
 'WDSP_ATTRIBUTES',
 'MXSPD',
 'GUST',
 'MAX',
 'MAX_ATTRIBUTES',
 'MIN',
 'MIN_ATTRIBUTES',
 'PRCP',
 'PRCP_ATTRIBUTES',
 'SNDP',
 'FRSHTT']

 **Count of member casual and rideable**

In [0]:
df_clean.groupBy("member_casual").count().show()
df_clean.groupBy("rideable_type").count().show()

+-------------+-------+
|member_casual|  count|
+-------------+-------+
|       member|3667790|
|       casual|1115786|
+-------------+-------+

+-------------+-------+
|rideable_type|  count|
+-------------+-------+
|electric_bike|3105929|
| classic_bike|1677647|
+-------------+-------+



In [0]:
df_clean.groupBy("TEMP").count().orderBy("TEMP").show()

+----+------+
|TEMP| count|
+----+------+
|63.3|     2|
|66.7|163574|
|66.9|  1339|
|68.1|156913|
|68.8|172067|
|69.7|151470|
|69.9|172804|
|70.7|171884|
|70.9|170945|
|71.1|144900|
|71.3|173002|
|71.4|172698|
|71.5|150113|
|72.5|144850|
|72.9|172719|
|73.3|172213|
|73.5|145235|
|73.6|150303|
|73.8|167253|
|74.2|170978|
+----+------+
only showing top 20 rows


**Distinct temp values**

In [0]:
df_clean.select("TEMP").distinct().orderBy("TEMP").show(100, truncate=False)

+----+
|TEMP|
+----+
|63.3|
|66.7|
|66.9|
|68.1|
|68.8|
|69.7|
|69.9|
|70.7|
|70.9|
|71.1|
|71.3|
|71.4|
|71.5|
|72.5|
|72.9|
|73.3|
|73.5|
|73.6|
|73.8|
|74.2|
|75.1|
|76.6|
|77.5|
|77.8|
|77.9|
|78.6|
|80.0|
|81.0|
|81.4|
|82.2|
|83.7|
+----+



**Calculated distance based on latitude and longitude**

In [0]:
from pyspark.sql.functions import col, radians, sin, cos, atan2, sqrt, when


df_clean = df_clean.withColumn(
    "distance_km",
    when(
        col("start_lat").isNull() | col("end_lat").isNull(),
        None
    ).otherwise(
        6371 * 2 * atan2(
            sqrt(
                sin((radians(col("end_lat")) - radians(col("start_lat"))) / 2) ** 2 +
                cos(radians(col("start_lat"))) * cos(radians(col("end_lat"))) *
                sin((radians(col("end_lng")) - radians(col("start_lng"))) / 2) ** 2
            ),
            sqrt(
                1 - (
                    sin((radians(col("end_lat")) - radians(col("start_lat"))) / 2) ** 2 +
                    cos(radians(col("start_lat"))) * cos(radians(col("end_lat"))) *
                    sin((radians(col("end_lng")) - radians(col("start_lng"))) / 2) ** 2
                )
            )
        )
    )
)

In [0]:
df_clean.select("distance_km").describe().show()

+-------+------------------+
|summary|       distance_km|
+-------+------------------+
|  count|           4768774|
|   mean|2.1606741015811908|
| stddev|17.829612699331186|
|    min|               0.0|
|    max| 8666.147811499419|
+-------+------------------+



In [0]:
from pyspark.sql.functions import round

df_clean.select(round("distance_km",2)).show()

+---------------------+
|round(distance_km, 2)|
+---------------------+
|                 1.89|
|                 1.36|
|                 2.09|
|                 3.01|
|                 4.03|
|                 4.21|
|                 1.23|
|                 5.08|
|                  0.5|
|                 6.88|
|                  9.3|
|                 0.79|
|                 3.76|
|                 2.75|
|                 3.28|
|                  3.2|
|                 2.11|
|                  0.5|
|                  5.8|
|                 1.68|
+---------------------+
only showing top 20 rows


In [0]:
import pandas
pdf = df_clean.limit(100000).toPandas()

In [0]:
pdf.corr()

---------------------------------------------------------------------------
ValueError                                Traceback (most recent call last)
File <command-8932979877315063>, line 1
----> 1 pdf.corr()

File /databricks/python/lib/python3.12/site-packages/pandas/core/frame.py:11049, in DataFrame.corr(self, method, min_periods, numeric_only)
  11047 cols = data.columns
  11048 idx = cols.copy()
> 11049 mat = data.to_numpy(dtype=float, na_value=np.nan, copy=False)
  11051 if method == "pearson":
  11052     correl = libalgos.nancorr(mat, minp=min_periods)

File /databricks/python/lib/python3.12/site-packages/pandas/core/frame.py:1993, in DataFrame.to_numpy(self, dtype, copy, na_value)
   1991 if dtype is not None:
   1992     dtype = np.dtype(dtype)
-> 1993 result = self._mgr.as_array(dtype=dtype, copy=copy, na_value=na_value)
   1994 if result.dtype is not dtype:
   1995     result = np.asarray(result, dtype=dtype)

File /databricks/python/lib/python3.12/site-packages/pandas/co